# Expérimentation de la méthode d'évaluation "Retrieval"

## Importation des bibliothèques

In [1]:
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import pytrec_eval
import json

/info/etu/m2/s2101703/miniconda3/envs/ETE/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-06 11:51:32.388058: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 11:51:32.440179: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-06 11:51:35.786502: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. Y

## Chargement du corpus et du modèle

In [2]:
newsgroups = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))
docs = newsgroups['data']

topic_model = BERTopic(calculate_probabilities=True)
topics, probs = topic_model.fit_transform(docs)

## Retrieval

In [3]:
topic_model.get_document_info(docs)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,\n\nI am sure some bashers of Pens fans are pr...,0,0_game_team_games_he,"[game, team, games, he, players, season, hocke...",[1992-93 Los Angeles Kings notes and game repo...,game - team - games - he - players - season - ...,0.288971,False
1,My brother is in the market for a high-perform...,4,4_card_monitor_video_drivers,"[card, monitor, video, drivers, vga, monitors,...",[Hello all.\n\tI am thinking about buying an e...,card - monitor - video - drivers - vga - monit...,0.139678,False
2,\n\n\n\n\tFinally you said what you dream abou...,-1,-1_to_the_of_is,"[to, the, of, is, and, you, in, for, it, that]","[\n\nBrian K., I am pleased with your honesty....",to - the - of - is - and - you - in - for - it...,0.685516,False
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,49,49_scsi_scsi2_scsi1_ide,"[scsi, scsi2, scsi1, ide, asynchronous, contro...",[wlsmith@valve.heart.rri.uwo.ca (Wayne Smith) ...,scsi - scsi2 - scsi1 - ide - asynchronous - co...,0.139445,False
4,1) I have an old Jasmine drive which I cann...,90,90_tape_backup_tapes_drive,"[tape, backup, tapes, drive, device, munroe, w...",[I have a Colorado Memory Systems Jumbo 250 ta...,tape - backup - tapes - drive - device - munro...,0.140558,False
...,...,...,...,...,...,...,...,...
18841,DN> From: nyeda@cnsvax.uwec.edu (David Nye)\nD...,23,23_cancer_medical_doctor_medicine,"[cancer, medical, doctor, medicine, disease, p...",[The biggest reason why the cost of medical ca...,cancer - medical - doctor - medicine - disease...,0.320454,False
18842,\nNot in isolated ground recepticles (usually ...,169,169_ground_grounding_conductor_neutral,"[ground, grounding, conductor, neutral, wire, ...","[\nNot according to the NEC nor the CEC, as ex...",ground - grounding - conductor - neutral - wir...,0.498710,False
18843,I just installed a DX2-66 CPU in a clone mothe...,82,82_fan_cpu_heat_sink,"[fan, cpu, heat, sink, fans, cooling, chip, ho...","[N(P>Just got a 66MHz 486DX2 system, and am co...",fan - cpu - heat - sink - fans - cooling - chi...,1.000000,False
18844,\nWouldn't this require a hyper-sphere. In 3-...,20,20_den_polygon_points_algorithm,"[den, polygon, points, algorithm, xxxx, sphere...","[\nSorry!! :-)\n\nCall the four points A, B, C...",den - polygon - points - algorithm - xxxx - sp...,1.000000,False


In [4]:
def getWordsTopics(topicsKey :int, model : BERTopic) -> list :
    return [word for word,_ in model.get_topic(topicsKey)]

In [15]:
def prepare_data(topic_model: BERTopic) -> tuple[dict, dict]:
    """
    Prépare le dictionnaire qrel et run pour la bibliothèque pytrec_eval
    topic_model: modèle BERTopic
    return: dictionnaire qrel et run
    """
    
    documentInfos = topic_model.get_document_info(docs=docs)
    documentInfos = documentInfos[documentInfos['Topic'] != -1]

    # Faire l'embedding de tous les documents d'un coup
    all_docs = documentInfos['Document'].tolist()
    docs_embeddings = np.array(
        topic_model.embedding_model.embed_documents(all_docs)
    ) # shape: (n_docs, embedding_dim)

    doc_ids = [str(doc) for doc in documentInfos.index]
    doc_topics = documentInfos['Topic'].tolist()
    
    topics = {k: v for k, v in topic_model.get_topics().items() if k != -1}
    qrel = dict()
    predictions = dict()

    for topic, words in topics.items():
        topic_id = str(topic)
        qrel[topic_id] = {}
        predictions[topic_id] = {}

        # Obtention des mots-clés pour un topic
        words = getWordsTopics(topic, topic_model)

        # Faire l'embedding des mots-clés
        embeddingMotsCles = topic_model.embedding_model.embed_words(
            words=[w for w in words]
        )
        
        # Calcul des scores
        scores = cosine_similarity(docs_embeddings, embeddingMotsCles).flatten()
        print(scores.shape)

        # Construction des dicts
        predictions[topic_id] = {
            doc_id: float(score)
            for doc_id, score in zip(doc_ids, scores)
        }
        qrel[topic_id] = {
            doc_id: 1 if t == topic else 0
            for doc_id, t in zip(doc_ids, doc_topics)
        }
        
    return qrel, predictions

In [16]:
def retrieval(topic_model: BERTopic) -> {}:
    """
    Calcule les métriques de "retrieval" en préparant le dictionnaire qrel et run à l'aide de la bibliothèque pytrec_eval.
    Métriques : 
        - Mean Average Precision (MAP)
        - Normalized Discounted Cumulative Gain (NDCG)
    topic_model: modèle BERTopic
    return: dictionnaire des métriques
    """
    
    qrel, predictions = prepare_data(topic_model=topic_model)

    evaluator = pytrec_eval.RelevanceEvaluator(qrel, {'map', 'ndcg'})
    results = evaluator.evaluate(predictions)
    
    return results

In [17]:
results = retrieval(topic_model=topic_model)

(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)
(121560,)


In [18]:
results

{'0': {'map': 0.15246553556199002, 'ndcg': 0.7639505708374479},
 '1': {'map': 0.04881487098838512, 'ndcg': 0.6176990387621362},
 '2': {'map': 0.04394504500079218, 'ndcg': 0.6030480555245518},
 '3': {'map': 0.036635408563420985, 'ndcg': 0.5811569799271044},
 '4': {'map': 0.032899463627277165, 'ndcg': 0.56744793368383},
 '5': {'map': 0.02826059807766187, 'ndcg': 0.5456100366783905},
 '6': {'map': 0.026817528478763127, 'ndcg': 0.5398406392553259},
 '7': {'map': 0.01831093295020994, 'ndcg': 0.49203324373833346},
 '8': {'map': 0.015710185093616574, 'ndcg': 0.4730454511968257},
 '9': {'map': 0.014050074470567978, 'ndcg': 0.45824497970082567},
 '10': {'map': 0.013521930786180617, 'ndcg': 0.45648411755482404},
 '11': {'map': 0.011082372874982661, 'ndcg': 0.43603307677040043},
 '12': {'map': 0.010512946555774989, 'ndcg': 0.42970927580360185},
 '13': {'map': 0.01329515507476768, 'ndcg': 0.44004161099161804},
 '14': {'map': 0.010938426408962762, 'ndcg': 0.42825675039698724},
 '15': {'map': 0.0081